# Router Stage 2 v3 - 6-way modality classifier, bone_fracture removed

**What changed vs v2:** 7 -> **6 classes** (`brain_mri, breast_histology, kidney_ct, pneumonia, retinal, skin_lesion`). Full retrain from ImageNet weights, same recipe and data mapping
(BreakHis for breast_histology, 12,000 cap per modality except breast), plain unweighted loss, group-aware splits.

Class order is alphabetical, so indices shift vs v2 (bone_fracture used to be index 0). The backend reads `class_names` from the checkpoint, so that is fine - but make sure nothing hard-codes a 7-class index.
In-distribution accuracy will again look near-perfect; that number already proved unreliable on the gatekeeper, so the real decision is Steps 7-8.

## 1. Setup

In [1]:
import sys, time, shutil, importlib
from pathlib import Path
ROOT = Path(r"E:\programming\CliniqAI")
sys.path.insert(0, str(ROOT / "scripts"))
import torch
import router_common as rc
importlib.reload(rc)            # picks up edits to router_common.py without a kernel restart
torch.backends.cudnn.benchmark = True
rc.set_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")
DATA_RAW   = ROOT / "data" / "raw"
MODELS_DIR = ROOT / "models"

torch 2.5.0+cu124 | device: cuda NVIDIA GeForce RTX 4050 Laptop GPU


## 2. Config

In [2]:
TAG        = "router_stage2_v3_no_bone"
FINAL_PATH = MODELS_DIR / f"{TAG}.pt"
CLASS_NAMES = list(rc.MODALITIES)

BATCH_SIZE = 64
NUM_WORKERS = 4
EPOCHS, PATIENCE = 20, 7
LR, WD = 1e-3, 1e-4
RESUME = True
print("CLASS_NAMES =", CLASS_NAMES)

CLASS_NAMES = ['brain_mri', 'breast_histology', 'kidney_ct', 'pneumonia', 'retinal', 'skin_lesion']


## 3. Build sample lists + group-aware splits

In [3]:
splits = rc.build_stage2_samples(DATA_RAW, CLASS_NAMES)
rc.print_split_table(splits, CLASS_NAMES)
rc.assert_no_group_overlap(splits)

  brain_mri          brain_tumor            found   7200  using   7200
  breast_histology   breast_histology_v2    found   7909  using   7909
  kidney_ct          kidney_stone           found  12446  using  12000
  pneumonia          pneumonia              found   4751  using   4751
  retinal            diabetic_retinopathy   found   3662  using   3662
  skin_lesion        skin_lesion            found  25331  using  12000

source                       class           train      val     test
brain_mri                    brain_mri        5760      720      720
breast_histology             breast_histology     6356      821      732
kidney_ct                    kidney_ct        9600     1200     1200
pneumonia                    pneumonia        3802      475      474
retinal                      retinal          2930      366      366
skin_lesion                  skin_lesion      9600     1200     1200
TOTAL                                        38048     4782     4692
  train class cou

## 4. Datasets, loaders, model

In [4]:
train_ds = rc.ListDataset(splits["train"], rc.train_transform(degrade_p=0.0))
val_ds   = rc.ListDataset(splits["val"],   rc.eval_transform())
test_ds  = rc.ListDataset(splits["test"],  rc.eval_transform())
train_loader = rc.make_loader(train_ds, BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
val_loader   = rc.make_loader(val_ds,   BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, persistent=True)
model = rc.build_model(len(CLASS_NAMES), pretrained=True)
print(len(train_ds), len(val_ds), len(test_ds))

38048 4782 4692


## 5. Train  (re-running this cell resumes from the best checkpoint)

In [5]:
history = rc.train_model(model, train_loader, val_loader, CLASS_NAMES, MODELS_DIR, TAG, DEVICE,
                         epochs=EPOCHS, lr=LR, weight_decay=WD, patience=PATIENCE, resume=RESUME)

epoch 1/20:   0%|          | 0/594 [00:12<?, ?it/s]

epoch  1/20 | train 0.0347/0.9925 | val 0.0077/0.9971 | lr 1.0e-03 | 4.7 min  * best


epoch 2/20:   0%|          | 0/594 [00:00<?, ?it/s]

epoch  2/20 | train 0.0183/0.9953 | val 0.0020/0.9994 | lr 1.0e-03 | 4.4 min  * best


epoch 3/20:   0%|          | 0/594 [00:00<?, ?it/s]

...

epoch 18/20 | train 0.0004/0.9999 | val 0.0001/1.0000 | lr 2.5e-04 | 3.6 min


epoch 19/20:   0%|          | 0/594 [00:00<?, ?it/s]

epoch 19/20 | train 0.0002/0.9999 | val 0.0000/1.0000 | lr 2.5e-04 | 3.7 min


epoch 20/20:   0%|          | 0/594 [00:00<?, ?it/s]

epoch 20/20 | train 0.0012/0.9998 | val 0.0000/1.0000 | lr 1.3e-04 | 3.6 min
Loaded best weights (val_loss 0.0000, epoch 14)


## 6. Save

In [6]:
rc.save_final(model, CLASS_NAMES, history, FINAL_PATH, extra={"excluded_modalities": ["bone_fracture"], "version": "v3"})
COPY_TO_BACKEND = False
if COPY_TO_BACKEND:
    (ROOT / "backend" / "models").mkdir(exist_ok=True)
    shutil.copy2(FINAL_PATH, ROOT / "backend" / "models" / FINAL_PATH.name)

Saved E:\programming\CliniqAI\models\router_stage2_v3_no_bone.pt  +  router_stage2_v3_no_bone.history.json   (round-trip load OK, classes=['brain_mri', 'breast_histology', 'kidney_ct', 'pneumonia', 'retinal', 'skin_lesion'])


## 7. Test-set breakdown + the two known confusion pairs

In [7]:
probs = rc.predict_probs(model, test_ds, DEVICE, num_workers=NUM_WORKERS)
cm = rc.report(probs, test_ds.samples, CLASS_NAMES, "STAGE 2 v3 TEST")
rc.pair_confusions(cm, CLASS_NAMES, [("kidney_ct", "brain_mri"), ("breast_histology", "skin_lesion")])
print("\nReminder: a clean 0 here already happened with v2 before it was checked on real images - it does not clear the pairs by itself.")

predict:   0%|          | 0/74 [00:12<?, ?it/s]


===== STAGE 2 v3 TEST: overall acc 1.0000  (n=4692) =====
Per-class recall:
  brain_mri          1.0000   (n=720)
  breast_histology   1.0000   (n=732)
  kidney_ct          1.0000   (n=1200)
  pneumonia          1.0000   (n=474)
  retinal            1.0000   (n=366)
  skin_lesion        1.0000   (n=1200)
Confusion matrix (rows = true, cols = predicted):
                  brain_mri breast_hi kidney_ct pneumonia   retinal skin_lesi
        brain_mri       720         0         0         0         0         0
 breast_histology         0       732         0         0         0         0
        kidney_ct         0         0      1200         0         0         0
        pneumonia         0         0         0       474         0         0
          retinal         0         0         0         0       366         0
      skin_lesion         0         0         0         0         0      1200
Per-source accuracy (worst first):
  brain_mri                    1.0000   (n=720)
  breast_histo

## 8. Real-world checks (these decide whether v3 gets deployed)
Create `data/raw/manual_sanity_check_stage2/<modality>/` with REAL (non-dataset) images, one subfolder per modality, named exactly:
`brain_mri, breast_histology, kidney_ct, pneumonia, retinal, skin_lesion`.
Most important: real **kidney_ct** + **brain_mri** (the 92.8% confusion) and a real **breast_histology** image (the 99.8% -> skin_lesion case).

In [8]:
SAN = DATA_RAW / "manual_sanity_check_stage2"
tot = ok = 0
for m in CLASS_NAMES:
    r = rc.sanity_folder(model, SAN / m, m, CLASS_NAMES, DEVICE, label=m)
    if r: ok += r[0]; tot += r[1]
print(f"\nSTAGE 2 real-image sanity: {ok}/{tot} passed" if tot else "\nNo real sanity images found - Stage 2 is NOT validated yet.")

[brain_mri] WARNING: E:\programming\CliniqAI\data\raw\manual_sanity_check_stage2\brain_mri is missing/empty -> this check proves nothing yet
[breast_histology] WARNING: E:\programming\CliniqAI\data\raw\manual_sanity_check_stage2\breast_histology is missing/empty -> this check proves nothing yet
[kidney_ct] WARNING: E:\programming\CliniqAI\data\raw\manual_sanity_check_stage2\kidney_ct is missing/empty -> this check proves nothing yet
[pneumonia] WARNING: E:\programming\CliniqAI\data\raw\manual_sanity_check_stage2\pneumonia is missing/empty -> this check proves nothing yet
[retinal] WARNING: E:\programming\CliniqAI\data\raw\manual_sanity_check_stage2\retinal is missing/empty -> this check proves nothing yet
[skin_lesion] WARNING: E:\programming\CliniqAI\data\raw\manual_sanity_check_stage2\skin_lesion is missing/empty -> this check proves nothing yet

No real sanity images found - Stage 2 is NOT validated yet.


## 9. Where do bone x-rays get routed now? (information only)
Stage 2 has no bone class any more. If the gatekeeper lets a bone x-ray through, this shows which specialist it will wrongly reach.

In [9]:
rc.probe_folder(model, DATA_RAW / "bone_fracture", CLASS_NAMES, DEVICE, n=300, title="bone x-rays (unseen modality)")

[bone x-rays (unseen modality)] folder missing/empty - skipped


## 10. Deploy checklist
1. Step 7 pairs + Step 8 real-image checks pass.
2. Copy `models/router_stage2_v3_no_bone.pt` to `backend/models/` if needed; point `ROUTER_STAGE2_PATH` at it.
3. Remove `bone_fracture` from the backend (`config.py` specialists, `KNOWN_CLASS_NAMES`, `schemas.py`, frontend labels) so the registry says 6/6 specialists.
4. Commit on `feature/router`.